# ChatBot 

1. Use different message types - `HumanMessage` and `AI Message`.
2. Maintain a full conversation history using both message types.
3. Use a **local LLM** via `LangChain's ChatOpenAI` pointed to **LM Studio**.
4. Create a sophisticated conversation loop.

### **Main Goal: create a form of memory for our Agent**

In [ ]:
from typing import TypedDict, List, Union
from langchain_core.messages import HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

In [ ]:
class AgentState(TypedDict):
    messages: List[Union[HumanMessage, AIMessage]] # With Union we can store either human or AI messages

#### Initialize the LLM

Although we use `ChatOpenAI`, setting `base_url` to point to **LM Studio** means all requests go to your **local server** — not to OpenAI's servers. The class is just an HTTP client that speaks the OpenAI API format, which LM Studio supports.

In [ ]:
# 100% local - connects to LM Studio, not OpenAI
llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",  # LM Studio doesn't validate this, any string works
    model="local-model"   # LM Studio ignores this, uses whatever model you have loaded
)

In [ ]:
def process(state: AgentState) -> AgentState:
    """
    This model will solve the request you input
    """
    response = llm.invoke(state["messages"])
    state["messages"].append(AIMessage(content = response.content))

    return state

In [ ]:
graph = StateGraph(AgentState)
graph.add_node("process", process)
graph.add_edge(START, "process")
graph.add_edge("process", END)
agent = graph.compile()

In [ ]:
converstation_history = [] # This will be our memory

user_input = input("Enter: ")

while user_input != "exit":
    converstation_history.append(HumanMessage(content = user_input))

    result = agent.invoke({"messages": converstation_history})

    print(result["messages"])
    converstation_history = result["messages"]
    user_input = input("Enter: ")

#### We can store the memory simply with a txt file

In [ ]:
with open("logging.txt", "w") as file:
    file.write("Your conversation Log: \n")

    for message in converstation_history:
        if isinstance(message, HumanMessage):
            file.write(f"You: {message.content} \n")
        if isinstance(message, AIMessage):
            file.write(f"AI: {message.content}")
    file.write("End of Conversation")

print("Conversation saved to logging.txt")